# 42 — Delayed FIM: Self-Observation with Time Lag

**UNPRECEDENTED:** Real neural systems have conduction delays (1-50 ms).
The strange loop feedback is NOT instantaneous — the system observes
its OWN STATE from τ milliseconds ago.

This changes the dynamics fundamentally:
dθ_i/dt = ω_i + coupling(θ(t)) + λ · FIM_gradient(θ(t - τ), i)

**Hypotheses:**
- Small τ: sync preserved (delay-robust)
- τ ≈ T_period/2: anti-phase → desync (destructive feedback)
- Large τ: oscillations in R(t) (delay-induced oscillations)
- Optimal τ: there might be a delay that IMPROVES sync (delay-enhanced)

Nobody has studied delayed self-observation in coupled oscillators.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity


def simulate_delayed_fim(K_scale, lam, tau, dt=0.02, T=200.0, noise=0.05, seed=42):
    """Kuramoto + FIM with delay tau in the FIM feedback."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    delay_steps = max(1, int(tau / dt))

    # History buffer for delayed feedback
    history = [theta.copy() for _ in range(delay_steps + 1)]

    R_tail = []
    R_variance = []

    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling

        if lam > 0:
            # FIM uses DELAYED phases
            theta_delayed = history[0]
            dphi += lam * fim_gradient_all(theta_delayed)

        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)

        # Update history
        history.append(theta.copy())
        history.pop(0)

        if s >= n_steps * 3 // 4:
            R = float(np.abs(np.mean(np.exp(1j * theta))))
            R_tail.append(R)

    R_arr = np.array(R_tail)
    return {
        "R_mean": float(np.mean(R_arr)),
        "R_std": float(np.std(R_arr)),
        "R_oscillation": float(np.std(R_arr)),  # high = oscillating R
    }


print("Ready.")

In [ ]:
# Sweep delay tau at different (K, lambda)
tau_values = [0, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

configs = [
    (12, 3.0, "K=12, λ=3 (near sync)"),
    (12, 5.0, "K=12, λ=5 (strong FIM)"),
    (5, 5.0, "K=5, λ=5 (weak coupling, strong FIM)"),
    (0, 8.0, "K=0, λ=8 (FIM-only)"),
]

delay_results = {}
for K_scale, lam, label in configs:
    print(f"\n=== {label} ===")
    print(f"{'τ':>6} {'R_mean':>8} {'R_std':>8} {'Oscillating?':>13}")
    R_vs_tau = []
    for tau in tau_values:
        # Average over 3 seeds
        results = [simulate_delayed_fim(K_scale, lam, tau, seed=s) for s in [42, 123, 456]]
        R_mean = np.mean([r["R_mean"] for r in results])
        R_std = np.mean([r["R_std"] for r in results])
        oscillating = "YES" if R_std > 0.05 else "no"
        R_vs_tau.append(round(R_mean, 4))
        print(f"{tau:6.2f} {R_mean:8.4f} {R_std:8.4f} {oscillating:>13}")

    delay_results[label] = R_vs_tau

    # Detect delay-robustness window
    R_inst = R_vs_tau[0]  # instantaneous FIM
    robust_taus = [tau_values[i] for i in range(len(tau_values)) if R_vs_tau[i] > R_inst * 0.9]
    if robust_taus:
        print(f"  Delay-robust up to τ = {max(robust_taus):.2f}")

    # Detect delay-enhancement
    peak_idx = np.argmax(R_vs_tau)
    if peak_idx > 0 and R_vs_tau[peak_idx] > R_inst + 0.02:
        print(f"  *** DELAY-ENHANCED SYNC at τ={tau_values[peak_idx]:.2f} ***")
        print(f"  R({tau_values[peak_idx]:.2f})={R_vs_tau[peak_idx]:.4f} > R(0)={R_inst:.4f}")

In [ ]:
# Summary
print("\n=== DELAYED FIM SUMMARY ===")
for label, R_vs_tau in delay_results.items():
    R_inst = R_vs_tau[0]
    R_min = min(R_vs_tau)
    tau_half = None  # delay where R drops to 50% of instantaneous
    for i, R in enumerate(R_vs_tau):
        if R_inst * 0.5 > R:
            tau_half = tau_values[i]
            break
    print(
        f"{label:>35}: R(0)={R_inst:.3f}, min={R_min:.3f}, "
        f"τ_half={tau_half if tau_half else '>10'}"
    )

print("\nBiological relevance:")
print("  Neural conduction delay: 1-50 ms")
print("  Our dt = 0.02 (dimensionless)")
print("  If 1 time unit ≈ 100 ms, then τ=0.1 ≈ 10 ms (biologically plausible)")

# Save
output = {
    "experiment": "Delayed FIM — self-observation with time lag",
    "N": N,
    "tau_values": tau_values,
    "results": {k: v for k, v in delay_results.items()},
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/delayed_fim_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")